In [2]:
import openpyxl,datetime
from openpyxl.styles import Font,PatternFill,Alignment

In [4]:
SRC_PATH = "input.xlsx"   
OUT_PATH = "Table_S2_corrected_windows.xlsx"        

In [6]:
wb = openpyxl.load_workbook(SRC_PATH, data_only=True)
ws = wb.active

storms = {}      # storm number -> list of (time, P, Vup, Vdown, SM50)
reference = {}   # (storm, "storm.j") -> reference row values (cols K..Q)
current = None

for row in list(ws)[1:]:          # skip header row
    v = [c.value for c in row]

    # A "Storm N" label in column A starts a new storm block
    if isinstance(v[0], str) and v[0].startswith("Storm "):
        current = int(v[0].split()[1])
        storms[current] = []

    # Time-series rows: column B is a datetime
    if current and isinstance(v[1], datetime.datetime):
        storms[current].append((
            v[1],                 # timestamp
            float(v[3] or 0),     # rain depth P (mm)
            float(v[4] or 0),     # Vup (mL)
            float(v[5] or 0),     # Vdown (mL)
            v[8],                 # SM50
        ))

    # Reference Table S2 rows: column J holds the RPE id (e.g. 3.2)
    if current and isinstance(v[9], (int, float)):
        reference[(current, str(v[9]))] = v[10:17]

# Expected RPE boundary times per storm (used as a sanity check)
expected = {
    1: [],
    2: ["09:34"],
    3: ["13:53", "15:08", "16:08"],
    4: ["07:30", "08:45", "09:30", "11:00"],
    5: ["18:22"],
    6: ["08:45"],
}

In [8]:
def limb_peak(p, start):
    """From index `start`, walk up the rising limb and return the peak index.
    Returns None if the series does not actually rise from `start`."""
    if start >= len(p) - 1:
        return start
    if p[start + 1] < p[start]:
        return None
    j = start + 1
    rise = p[j] > p[start]
    while j + 1 < len(p) and p[j + 1] >= p[j]:
        rise |= p[j + 1] > p[j]
        j += 1
    return j if rise else None


def boundaries(arr, PT=0.75, mult=1.25):
    """Return indices of the troughs that split a storm into separate RPEs.

    A trough i (followed by a new rising limb to peak nxt) is a boundary if:
      - p[i] < PT and p[nxt] > PT, or
      - p[i] >= PT and p[nxt] >= mult * p[current peak]
    """
    p = [a[1] for a in arr]
    first = next((i for i, val in enumerate(p) if val > 0), None)
    if first is None:
        return []

    peak = limb_peak(p, first) or first
    cur = peak
    result = []

    while cur < len(p) - 1:
        found = None
        for i in range(cur + 1, len(p) - 1):
            if p[i] <= p[i - 1] and p[i + 1] > p[i]:   # local trough
                nxt = limb_peak(p, i)
                if nxt is not None:
                    found = (i, nxt)
                    break
        if found is None:
            break

        i, nxt = found
        if (p[i] < PT and p[nxt] > PT) or (p[i] >= PT and p[nxt] >= mult * p[cur]):
            result.append(i)
        cur = nxt

    return result

In [10]:
out = openpyxl.Workbook()
o = out.active
o.title = "Calculated Table S2"

headers = [
    "Storm", "RPE", "Start", "End", "Rain depth (mm)", "Vup (mL)", "Vdown (mL)",
    "SM50 pre", "C", "Ebs", "Rc",
    "Reference Vup", "Difference Vup", "Reference Vdown", "Difference Vdown",
]
o.append(headers)

checks = []

for storm, arr in storms.items():
    # --- RPE boundaries ---
    b = boundaries(arr)
    times = [arr[i][0].strftime("%H:%M") for i in b]
    print("Storm", storm, "boundaries", times, "expected", expected[storm])
    assert times == expected[storm], f"Storm {storm}: boundary mismatch"

    # --- storm start / end (zero-rain rows bracketing the event) ---
    p = [a[1] for a in arr]
    positive = [i for i, x in enumerate(p) if x > 0]
    start = positive[0] - 1
    assert start >= 0 and p[start] == 0, f"Storm {storm}: no zero row before rain"
    last = positive[-1]
    end = next(i for i in range(last + 1, len(p)) if p[i] == 0)

    starts = [start] + b
    ends = b + [end]

    for j, (a, z) in enumerate(zip(starts, ends), 1):
        # Shared boundary rows are split 50/50 between adjacent RPEs
        weights = [
            0.5 if ((j > 1 and i == a) or (j < len(starts) and i == z)) else 1.0
            for i in range(a, z + 1)
        ]
        sums = [
            sum(arr[i][k] * weights[i - a] for i in range(a, z + 1))
            for k in [1, 2, 3]
        ]
        P, vu, vd = sums

        sm = arr[a][4]
        sm = float(sm) if sm is not None else None

        C = vu / (P * 1629 * 1000) if P else None      # runoff coefficient
        E = (vu - vd) / vu if vu else None              # bioswale efficiency
        R = C * E if C is not None and E is not None else None

        ref = reference.get((storm, f"{storm}.{j}"))
        rv = ref[1] if ref else None
        rd = ref[2] if ref else None

        o.append([
            storm, f"{storm}.{j}", arr[a][0], arr[z][0],
            P, vu, vd, sm, C, E, R,
            rv, vu - rv if rv is not None else None,
            rd, vd - rd if rd is not None else None,
        ])
        checks.append((
            storm, j,
            vu - rv if rv is not None else None,
            vd - rd if rd is not None else None,
        ))

In [12]:
for c in o[1]:
    c.font = Font(bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor="153A60")

for col in "CDE":
    for cell in o[col][1:]:
        cell.number_format = "yyyy-mm-dd hh:mm"
for col in "FG":
    for cell in o[col][1:]:
        cell.number_format = "#,##0.00"
for col in "IJK":
    for cell in o[col][1:]:
        cell.number_format = "0.0000"

for col in o.columns:
    letter = col[0].column_letter
    o.column_dimensions[letter].width = 24 if letter in "CD" else 19

o.freeze_panes = "A2"
o.auto_filter.ref = o.dimensions
out.save(OUT_PATH)

print("CHECKS  (storm, RPE, ΔVup, ΔVdown)")
for x in checks:
    print(x)


CHECKS  (storm, RPE, ΔVup, ΔVdown)
